🧩 Cell 1 – Install Required Packages

In [10]:
# =========================================================
# 🕉️ MODULE 2: AI-Based Sanskrit Sandhi Splitter
# Ayurvedic Text Analysis Pipeline
# =========================================================
# 📦 Install Required Packages (Colab-compatible versions)
!pip install -q torch transformers sentencepiece protobuf pandas indic-transliteration tabulate accelerate

print("✅ All packages installed successfully!")


✅ All packages installed successfully!


🧩 Cell 2 – Imports & Environment Setup

In [11]:
# =========================================================
# IMPORT LIBRARIES AND SETUP
# =========================================================
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, T5ForConditionalGeneration, MT5ForConditionalGeneration
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from IPython.display import display, HTML
import warnings, re

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")
print(f"🐍 PyTorch version: {torch.__version__}")


🔧 Using device: cpu
🐍 PyTorch version: 2.8.0+cu126


🧩 Cell 3 – Define the Sanskrit Sandhi AI Class

In [12]:
# =========================================================
# DEFINE THE SANSKRIT SANDHI SPLITTER CLASS
# =========================================================

class SanskritSandhiAI:
    """AI-based Sanskrit Sandhi Splitter using Transformer models."""

    def __init__(self):
        self.device = device
        self.model, self.tokenizer, self.model_name, self.model_type = None, None, None, None
        self._load_best_model()

    def _load_best_model(self):
        models_to_try = [
            ("ai4bharat/IndicBART", "mbart"),
            ("google/mt5-small", "mt5"),
            ("google/byt5-small", "byt5"),
            ("google/t5-small", "t5"),
        ]

        for model_id, mtype in models_to_try:
            try:
                print(f"📚 Attempting to load: {model_id}")
                self.tokenizer = AutoTokenizer.from_pretrained(model_id)
                if mtype == "mt5":
                    self.model = MT5ForConditionalGeneration.from_pretrained(model_id)
                else:
                    self.model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
                self.model = self.model.to(self.device).eval()
                self.model_name, self.model_type = model_id, mtype
                print(f"✅ Loaded model: {model_id}")
                break
            except Exception as e:
                print(f"⚠️ Could not load {model_id}: {str(e)[:100]}")
                continue

        if self.model is None:
            raise RuntimeError("❌ No models could be loaded. Check internet connection.")

    def _prepare_input(self, text):
        prefix = {"mt5": "split sandhi: ", "byt5": "sandhi: "}.get(self.model_type, "separate: ")
        return prefix + text.strip()

    def _apply_sandhi_rules(self, text):
        rules = {'ोऽ': 'ः अ', 'ाऽ': 'ा अ', 'एऽ': 'ए अ', 'द्व': 'त् व', 'ज्ज': 'त् ज', 'च्च': 'त् च'}
        for k, v in rules.items():
            text = text.replace(k, v)
        return text

    def split_sandhi(self, text, use_beam=True):
        if not text: return text
        inputs = self.tokenizer(
            self._prepare_input(text),
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128,
            return_token_type_ids=False
            ).to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=128, num_beams=5 if use_beam else 1)
        pred = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        if not pred or pred == text:
            pred = self._apply_sandhi_rules(text)
        return ' '.join(pred.split())

    def batch_split(self, texts):
        return [self.split_sandhi(t) for t in texts]


🧩 Cell 4 – Initialize the Model

In [13]:
print("="*60)
print("🚀 Initializing Sanskrit Sandhi AI Model...")
print("="*60)

sandhi_ai = SanskritSandhiAI()
print(f"\n✅ Model Loaded Successfully: {sandhi_ai.model_name}")
print(f"🎯 Ready for Sanskrit Sandhi Splitting!")


🚀 Initializing Sanskrit Sandhi AI Model...
📚 Attempting to load: ai4bharat/IndicBART
✅ Loaded model: ai4bharat/IndicBART

✅ Model Loaded Successfully: ai4bharat/IndicBART
🎯 Ready for Sanskrit Sandhi Splitting!


🧩 Cell 5 – Helper Functions

In [14]:
# =========================================================
# HELPER FUNCTIONS
# =========================================================
def to_iast(text):
    try: return transliterate(text, sanscript.DEVANAGARI, sanscript.IAST)
    except: return text

def display_result(input_text, output_text):
    print("\n" + "="*60)
    print("📝 INPUT:  ", input_text)
    print("✂️ SPLIT:  ", output_text)
    print("📖 IAST:   ", to_iast(output_text))
    print("="*60)


🧩 Cell 6 – Interactive User Input

In [15]:
# =========================================================
# INTERACTIVE INPUT
# =========================================================
def process_user_input():
    user_text = input("\nEnter a Sanskrit sentence (press Enter for demo): ").strip()
    if not user_text:
        user_text = "रामोऽस्ति"
        print(f"📌 Using demo text: {user_text}")
    split_text = sandhi_ai.split_sandhi(user_text)
    display_result(user_text, split_text)
    return user_text, split_text

user_input, user_output = process_user_input()



Enter a Sanskrit sentence (press Enter for demo): रामोऽस्ति

📝 INPUT:   रामोऽस्ति
✂️ SPLIT:   | separate: रामोऽस्ति | रामोऽस्तिस्ति | | | | | | | |ो रामोऽस्ति | रामोऽस्ति | | | | | | | | | | | | | | | | |ो रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | | | | | | | | | | | | | | | | |ो रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | |ो रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | रामोऽ[CLS]
📖 IAST:    | separate: rāmo'sti | rāmo'stisti | | | | | | | |o rāmo'sti | rāmo'sti | | | | | | | | | | | | | | | | |o rāmo'sti | rāmo'sti | rāmo'sti | | | | | | | | | | | | | | | | |o rāmo'sti | rāmo'sti | rāmo'sti | |o rāmo'sti | rāmo'sti | rāmo'sti | rāmo'sti | rāmo'sti | rāmo'[CLS]


🧩 Cell 7 – Batch Processing (Ayurvedic Examples)

In [16]:
# =========================================================
# PROCESS AYURVEDIC EXAMPLES
# =========================================================
ayurvedic_examples = [
    "रामोऽस्ति", "हरिद्रा ज्वरं नाशयति", "आमलकी पित्तं शमयति",
    "गुडुची कासं उपयुज्यते", "अश्वगन्धा तनावं नाशयति",
    "त्रिफला पित्तम् शमयति", "तुलसी कफं नाशयति",
    "शुण्ठी वातं निवारयति", "निम्बो रक्तं शोधयति", "ब्राह्मी स्मृतिं वर्धयति"
]

results = []
for i, txt in enumerate(ayurvedic_examples, 1):
    split_txt = sandhi_ai.split_sandhi(txt)
    results.append({'#': i, 'Input': txt, 'Predicted Split': split_txt, 'IAST': to_iast(split_txt)})
    print(f"[{i}] {txt} → {split_txt}")


[1] रामोऽस्ति → | separate: रामोऽस्ति | रामोऽस्तिस्ति | | | | | | | |ो रामोऽस्ति | रामोऽस्ति | | | | | | | | | | | | | | | | |ो रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | | | | | | | | | | | | | | | | |ो रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | |ो रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | रामोऽस्ति | रामोऽ[CLS]
[2] हरिद्रा ज्वरं नाशयति → | separate: हरिद्रा ज्वरं नाशयतितितितितितितितितितितिति हरिद्रा ज्वरं हरिद्रा ज्वरं नाशयति स्वास्थ्य स्वास्थ्य स्वास्थ्य स्वास्थ्य स्वास्थ्य स्वास्थ्य स्वास्थ्य स्वास्थ्य | | separate: हरिद्रा ज्वरं हरिद्रा ज्वरं नाशयतितितितिति | | | separate: हरिद्रा ज्वरं हरिद्रा ज्वरं हरिद्रा ज्वरं नाशयतिति | | | | | | | |द्रा ज्वरं नाशयति | | | | | हरिद्रा ज्वरं हरिद्रा ज्वरं नाशयति | | | | | | हरिद्रा ज्वरं नाशयति | | |[CLS]
[3] आमलकी पित्तं शमयति → दूध separate: आमलकी पित्तं शमयतितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितितिति आमलकी पित्तं शमयति शमयतितितितितितितितितितितितितितितितितिति दूध: आमलकी पित्तं शमयति दूधं शमयति दूधं शमयति दूध दूध दूध द

🧩 Cell 8 – Display Results

In [17]:
# =========================================================
# DISPLAY RESULTS TABLE
# =========================================================
results_df = pd.DataFrame(results)
display(HTML("<h3>🔤 Sanskrit Sandhi Splitting Results</h3>"))
display(results_df)


,#,Input,Predicted Split,IAST
0,1,रामोऽस्ति,| separate: रामोऽस्ति | रामोऽस्तिस्ति | | | | ...,| separate: rāmo'sti | rāmo'stisti | | | | | |...
1,2,हरिद्रा ज्वरं नाशयति,| separate: हरिद्रा ज्वरं नाशयतितितितितितितिति...,| separate: haridrā jvaraṃ nāśayatitititititit...
2,3,आमलकी पित्तं शमयति,दूध separate: आमलकी पित्तं शमयतितितितितितितिति...,dūdha separate: āmalakī pittaṃ śamayatitititit...
3,4,गुडुची कासं उपयुज्यते,clarified separate: गुडुची कासं उपयुज्यते एवं ...,clarified separate: guḍucī kāsaṃ upayujyate ev...
4,5,अश्वगन्धा तनावं नाशयति,अ्यान्ड separate: अश्वगन्धा तनावं नाशयतितितिति...,ayānḍa separate: aśvagandhā tanāvaṃ nāśayatiti...
5,6,त्रिफला पित्तम् शमयति,हिंदू separate: त्रिफला पित्तम् शमयतितितितितित...,hiṃdū separate: triphalā pittam śamayatitititi...
6,7,तुलसी कफं नाशयति,हिंदू separate: तुलसी कफं नाशयति हिंदू हिंदू ह...,hiṃdū separate: tulasī kaphaṃ nāśayati hiṃdū h...
7,8,शुण्ठी वातं निवारयति,৷ separate: शुण्ठी वातं निवारयतितितितितितितिति...,৷ separate: śuṇṭhī vātaṃ nivārayatitititititit...
8,9,निम्बो रक्तं शोधयति,ख़ून separate: निम्बो रक्तं शोधयतितितितितितिति...,k͟hūna separate: nimbo raktaṃ śodhayatitititit...
9,10,ब्राह्मी स्मृतिं वर्धयति,हिंदू separate: ब्राह्मी स्मृतिं वर्धयतितितिति...,hiṃdū separate: brāhmī smṛtiṃ vardhayatitititi...


🧩 Cell 9 – Save & Pipeline Integration

In [18]:
# =========================================================
# SAVE OUTPUTS & PIPELINE FUNCTION
# =========================================================
results_df.to_csv("sandhi_split_results.csv", index=False, encoding="utf-8")
results_df.to_json("sandhi_split_results.json", orient="records", force_ascii=False, indent=2)

def pipeline_ready_splitter(text):
    split_text = sandhi_ai.split_sandhi(text)
    return {'original': text, 'split': split_text, 'iast': to_iast(split_text), 'module': 'Sandhi_Splitter_AI_v1.0'}

print("💾 Saved results as CSV and JSON.")


💾 Saved results as CSV and JSON.


🧩 Cell 10 – Final Summary

In [19]:
# =========================================================
# FINAL SUMMARY
# =========================================================
display(HTML(f"""
<div style='background-color:#4CAF50;color:white;padding:20px;border-radius:10px;text-align:center;'>
<h2>✅ Sanskrit Sandhi Splitter - Ready!</h2>
<p>Module 2 of Ayurvedic Text Analysis Pipeline</p>
<p>Model Used: {sandhi_ai.model_name}</p>
<p>Device: {device}</p>
<p>Successfully Processed {len(results)} Sentences</p>
</div>
"""))
